# P1 model cost scenarios

Executed illustrative calculation, 2026-10-19. No API calls. Prices are from the dated companion model catalog. The 68,011 figure is a non-deduplicated file count, not a measured eligible-asset count. These tables compare unit prices under fixed token assumptions; they do not predict actual model usage or quality. No chart is needed for this small exact-value lookup.

Sources: [Ark prices](https://docs.volcengine.com/docs/ark/model-pricing), [model list](https://docs.volcengine.com/docs/ark/model-list). Run from the project root or this notebook directory. Standard library only.

In [1]:
from pathlib import Path
from decimal import Decimal
import json
catalog_path = Path('docs/evaluation/model-catalog-2026-10-19.json')
if not catalog_path.exists():
    catalog_path = Path('model-catalog-2026-10-19.json')
catalog = json.loads(catalog_path.read_text())
input_tokens, output_tokens, file_count = 4000, 600, 68011
rows = []
for model in catalog['models']:
    cost = (Decimal(str(model['input_price'])) * input_tokens + Decimal(str(model['output_price'])) * output_tokens) / 1000000
    rows.append({'model': model['name'], 'per_video_cny': str(cost),
                 'per_1000_videos_cny': str(cost * 1000),
                 'per_68011_files_cny': str(cost * file_count)})
assert sum([25, 200, 80, 320, 80]) == 705
assert rows[0]['per_video_cny'] == '0.002'
print('ILLUSTRATIVE: 4000 uncached non-audio input tokens + 600 output tokens; not measured usage.')
for row in rows:
    print(f"{row['model']:<16} | per video CNY {float(row['per_video_cny']):.5f} | 68011 files CNY {float(row['per_68011_files_cny']):.2f}")
scenario = {'scenario_only': True, 'input_tokens': input_tokens, 'output_tokens': output_tokens,
            'non_deduplicated_file_count': file_count, 'rows': rows,
            'caveats': ['Not measured costs', 'Assumes equal tokens only for unit-price comparison',
                        'Excludes cache, reasoning beyond assumed output, failures, retries, preprocessing, storage and labor',
                        'Actual content duration and eligibility distribution unknown']}
(catalog_path.parent / 'cost-scenarios.json').write_text(json.dumps(scenario, ensure_ascii=False, indent=2))


ILLUSTRATIVE: 4000 uncached non-audio input tokens + 600 output tokens; not measured usage.
Seed 2.0 Mini    | per video CNY 0.00200 | 68011 files CNY 136.02
Seed 2.0 Lite    | per video CNY 0.00456 | 68011 files CNY 310.13
Seed 2.0 Pro     | per video CNY 0.02240 | 68011 files CNY 1523.45
Seed 2.1 Lite    | per video CNY 0.00482 | 68011 files CNY 327.81
Seed 2.1 Turbo   | per video CNY 0.02100 | 68011 files CNY 1428.23
Seed 2.1 Pro     | per video CNY 0.04200 | 68011 files CNY 2856.46


At the fixed 4,000/600 assumption, 2.1 Lite costs CNY 0.00482 per file versus 2.0 Lite CNY 0.00456 and Mini CNY 0.002. This does not establish which model is cheapest per accepted annotation. Measure model-specific usage, failures, corrections and quality first. Actual full-library costs require representative duration and content eligibility distributions.